# Local Blockchair Data Exploration

Dieses Notebook zeigt, wie man lokal heruntergeladene Blockchair-Daten verwendet.

## Voraussetzungen

1. Daten heruntergeladen mit:
   ```bash
   python scripts/download_blockchair.py --year 2021 --output /Volumes/MySSD/bitcoin_data --remove-gz
   ```

2. PySpark installiert (via requirements.txt)

## Datenquellen-Modus

Dieses Notebook kann mit 3 verschiedenen Datenquellen arbeiten:
- **Demo:** Kleine Beispieldaten (kein Setup nötig)
- **BigQuery:** Google Cloud (benötigt Credentials)
- **Blockchair Local:** Lokal heruntergeladene Daten (benötigt Download)

## 1. Setup und Konfiguration

In [ ]:
# Standard Libraries
import os
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# PySpark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, avg, sum as spark_sum

# Projekt-Pfad
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(project_root))

# Plot-Konfiguration
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100

print(f"Projektverzeichnis: {project_root}")

In [ ]:
# ============================================================================
# KONFIGURATION: Wähle Datenquelle
# ============================================================================

# Option 1: Demo-Daten (klein, schnell, kein Setup)
USE_DEMO = False

# Option 2: BigQuery (Cloud, vollständig, benötigt Credentials)
USE_BIGQUERY = False

# Option 3: Lokale Blockchair-Daten (SSD, Jahr 2021)
USE_BLOCKCHAIR = True
BLOCKCHAIR_DATA_DIR = "/Volumes/MySSD/bitcoin_data/extracted"  # ANPASSEN!

# Zeitraum für Analyse (nur relevant für BigQuery/Blockchair)
START_DATE = "2021-01-01"
END_DATE = "2021-01-07"  # Erste Woche Januar 2021

print(f"Datenmodus: ", end="")
if USE_DEMO:
    print("Demo-Daten")
elif USE_BIGQUERY:
    print("BigQuery")
elif USE_BLOCKCHAIR:
    print(f"Blockchair Local ({BLOCKCHAIR_DATA_DIR})")
    print(f"Zeitraum: {START_DATE} bis {END_DATE}")

## 2. SparkSession initialisieren

In [ ]:
# Spark konfigurieren
spark = SparkSession.builder \
    .appName("Bitcoin Local Data Exploration") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print(f"✅ Spark Session gestartet")
print(f"   Version: {spark.version}")
print(f"   App Name: {spark.sparkContext.appName}")

## 3. Daten laden

In [ ]:
if USE_BLOCKCHAIR:
    # Import Blockchair Loader
    from scripts.load_blockchair_data import BlockchairDataLoader
    
    # Initialisiere Loader
    loader = BlockchairDataLoader(BLOCKCHAIR_DATA_DIR, spark=spark)
    
    print(f"\n{'='*70}")
    print(f"LOADING BLOCKCHAIR DATA")
    print(f"{'='*70}\n")
    
    # Lade Transaktionen
    df_transactions = loader.load_transactions(START_DATE, END_DATE, filter_coinbase=True)
    
    # Lade Blocks
    df_blocks = loader.load_blocks(START_DATE, END_DATE)
    
    # Lade Outputs
    df_outputs = loader.load_outputs(START_DATE, END_DATE)
    
    print(f"\n✅ Alle Daten geladen")
    
elif USE_BIGQUERY:
    # BigQuery Code (wie in 01_data_exploration.ipynb)
    from google.cloud import bigquery
    client = bigquery.Client()
    
    query = f"""
    SELECT *
    FROM `bigquery-public-data.crypto_bitcoin.transactions`
    WHERE DATE(block_timestamp) BETWEEN '{START_DATE}' AND '{END_DATE}'
        AND is_coinbase = FALSE
    LIMIT 100000
    """
    
    df_transactions_pd = client.query(query).to_dataframe()
    df_transactions = spark.createDataFrame(df_transactions_pd)
    
elif USE_DEMO:
    # Demo-Daten erstellen
    demo_data = [
        (1, "hash1", 2, 1, 50000000, 49500000, 500000),
        (2, "hash2", 3, 2, 75000000, 74200000, 800000),
        (3, "hash3", 1, 1, 10000000, 9800000, 200000),
    ]
    
    df_transactions = spark.createDataFrame(
        demo_data,
        ["id", "hash", "input_count", "output_count", "input_total", "output_total", "fee"]
    )
    
    print("✅ Demo-Daten erstellt")

## 4. Erste Exploration

In [ ]:
# Schema anzeigen
print("📋 Transaction Schema:\n")
df_transactions.printSchema()

In [ ]:
# Erste Zeilen
print("📊 Sample Transactions:\n")
df_transactions.select(
    "hash", "input_count", "output_count", "fee"
).show(10, truncate=False)

In [ ]:
# Basis-Statistiken
print("📈 Basic Statistics:\n")

total = df_transactions.count()
print(f"Total Transactions: {total:,}")

stats = df_transactions.select(
    avg("input_count").alias("avg_inputs"),
    avg("output_count").alias("avg_outputs"),
    avg("fee").alias("avg_fee")
).collect()[0]

print(f"Average Inputs:  {stats['avg_inputs']:.2f}")
print(f"Average Outputs: {stats['avg_outputs']:.2f}")
print(f"Average Fee:     {stats['avg_fee']/100000000:.8f} BTC")

## 5. Multi-Input-Analyse (Entity-Clustering Relevanz)

In [ ]:
# Multi-Input Distribution
print("🔗 Input Count Distribution:\n")

input_dist = df_transactions.groupBy("input_count") \
    .agg(count("*").alias("transaction_count")) \
    .orderBy("input_count") \
    .limit(20)

input_dist.show()

In [ ]:
# Multi-Input Ratio berechnen
single_input = df_transactions.filter(col("input_count") == 1).count()
multi_input = df_transactions.filter(col("input_count") > 1).count()
total = df_transactions.count()

print(f"{'='*70}")
print(f"MULTI-INPUT ANALYSIS")
print(f"{'='*70}")
print(f"Single-Input (1 address):  {single_input:,} ({single_input/total*100:.1f}%)")
print(f"Multi-Input (≥2 addresses): {multi_input:,} ({multi_input/total*100:.1f}%)")
print(f"Total:                     {total:,}")
print(f"{'='*70}")
print(f"\n💡 {multi_input/total*100:.1f}% der Transaktionen sind für Entity-Clustering nutzbar!")

## 6. Visualisierung

In [ ]:
# Konvertiere für Plotting zu Pandas
input_dist_pd = input_dist.toPandas()

# Plot
plt.figure(figsize=(12, 6))
plt.bar(input_dist_pd['input_count'], input_dist_pd['transaction_count'], 
        color='steelblue', edgecolor='black', linewidth=0.5)
plt.xlabel('Input Count', fontsize=11)
plt.ylabel('Number of Transactions (log scale)', fontsize=11)
plt.title(f'Input Count Distribution ({START_DATE} to {END_DATE})', fontsize=12)
plt.yscale('log')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("- Single-input transactions dominieren (typisches Pattern)")
print("- Multi-input transactions werden seltener mit steigender Input-Anzahl")
print("- Power-Law-Verteilung sichtbar")

## 7. Für Entity-Clustering relevante Transaktionen extrahieren

In [ ]:
if USE_BLOCKCHAIR:
    # Verwende Loader-Funktion
    df_multi_input = loader.get_multi_input_transactions(
        START_DATE, END_DATE,
        min_inputs=2,
        max_inputs=50  # Filtert Exchanges
    )
else:
    # Manuell filtern
    df_multi_input = df_transactions.filter(
        (col("input_count") >= 2) & (col("input_count") <= 50)
    )
    
    print(f"🔗 Multi-Input Transactions (2-50 inputs): {df_multi_input.count():,}")

# Beispiele anzeigen
print("\nSample Multi-Input Transactions:\n")
df_multi_input.select("hash", "input_count", "output_count", "fee").show(10, truncate=False)

## 8. SQL Queries (Optional)

In [ ]:
# Erstelle Temp View für SQL
df_transactions.createOrReplaceTempView("transactions")

print("✅ SQL View 'transactions' erstellt")
print("\nBeispiel Query:\n")

# SQL Query ausführen
result = spark.sql("""
    SELECT 
        input_count,
        COUNT(*) as tx_count,
        AVG(fee) / 100000000 as avg_fee_btc
    FROM transactions
    WHERE input_count BETWEEN 1 AND 10
    GROUP BY input_count
    ORDER BY input_count
""")

result.show()

## 9. Zusammenfassung

### Was haben wir erreicht?

1. ✅ Lokale Blockchair-Daten erfolgreich geladen
2. ✅ PySpark DataFrames erstellt und exploriert
3. ✅ Multi-Input-Transaktionen identifiziert (~25-40%)
4. ✅ SQL-Queries auf lokalen Daten möglich

### Nächste Schritte

Diese Daten können nun verwendet werden für:
- **Entity-Clustering** (Notebook 02): GraphFrames Connected Components
- **Whale-Detection** (Notebook 03): UTXO-Aggregation
- **Zeitreihen-Analyse** (Notebook 04): Whale-Movements über Zeit

### Datenquellen-Vergleich

| Datenquelle | Vorteile | Nachteile |
|------------|----------|----------|
| **Demo** | Schnell, kein Setup | Unrealistisch klein |
| **BigQuery** | Vollständig, aktuell | Cloud, Kosten, Quotas |
| **Blockchair Local** | Schnell, keine Kosten, offline | Nur 2021, großer Download |

**Empfehlung:** Blockchair Local für Entwicklung/Testing, BigQuery für finale Analyse mit vollem Dataset.